# OpenMate interactive demo

This notebook walks through the RAG and MCP sides of OpenMate from inside Jupyter.

It uses the IPython magics shipped in ``openmate.jupyter_magic`` — a thin wrapper
over ``rag/`` and ``openmate.adapters.tools.mcp_client``. Neither Chroma nor MCP
ship official magics; we wrote our own so the notebook surface stays in sync
with the rest of OpenMate.

> Requires the `rag` and `mcp` extras (and a Gmail OAuth client if you want to
> point at the real Gmail server). See `.env.example`.


In [1]:
%load_ext openmate.jupyter_magic

OpenMate Jupyter extension loaded.
  RAG:  %rag_ingest <path>    %rag_query <question> [--k N]
  MCP:  %mcp_add <name> <cmd>     %mcp_list <name>    %mcp_call <name> <tool> <json>


## 1. RAG — ingest and query

Defaults come from the same env vars the CLI uses (`RAG_STORE`, `RAG_DB`,
`RAG_COLLECTION`, `RAG_EMBEDDER`). Run `python -m rag.cli --help` for the
full list, or just override inline by editing your shell env.

### 1a. Ingest a directory

In [2]:
%rag_ingest docs/


ingested IngestReport(documents=17, chunks=293, sources=['01-domain-model-and-kernel.md', '02-agent-loop-and-runtime.md', '03-model-port-and-providers.md', '04-tools-and-mcp.md', '05-planning-and-reasoning.md', '06-memory-and-state.md', '07-retrieval-rag.md', '08-multi-agent-orchestration.md', '09-context-engineering.md', '10-safety-and-guardrails.md', '11-observability-and-evaluation.md', '12-production-and-reliability.md', '13-framework-interoperability.md', '14-skills.md', 'README.md', 'architecture.md', 'jupyter-magics.md']) chunks from 'docs/' into store=InMemoryVectorStore


The cell above walks `docs/`, chunks each file with `FixedWindowChunker`,
embeds with the configured embedder, and writes into the configured store.
You'll see something like `ingested 142 chunks from 'docs/' into store=ChromaVectorStore`.

### 1b. Query it

In [3]:
docs = %rag_query explain the agent loop
docs[:2]   # show the top two as plain dicts


[1] score=0.454 · source=14-skills.md
der`. Wire the provider into an agent via `assemble()` ([02](02-agent-loop-and-runtime.md)); cards appear in the system prompt and `load_skill` injects a body. **PoC acceptance:** with three skills on disk, the agent loads exactly the relevant one for a task, loads none for an unrelated task, and the loaded body appears in context. --- ## Phase 1 — Bundled resources & scripts (level-3 disclosure) A skill is more than prose — it can carry reference files and runnable scripts. - **Resources:** `Skill.render()` appends an index of bundled files; the model reads one via a `read_file`/resource tool only when needed (keeps the body small). - **Scripts:** a skill bundles a script it runs through the `ShellTool` ([04](04-tools-and-mcp.md)) — e.g., a batch summarizer — so procedures can include deterministic code, not just instructions. **Worked example — a skill on disk:** ``` skills/email/triage-inbox/ ├── SKILL.md # frontmatter: name: triage-inbox · desc

[{'id': '14-skills.md#6',
  'score': 0.4541341396153042,
  'text': 'der`. Wire the provider into an agent via `assemble()` ([02](02-agent-loop-and-runtime.md)); cards appear in the system prompt and `load_skill` injects a body. **PoC acceptance:** with three skills on disk, the agent loads exactly the relevant one for a task, loads none for an unrelated task, and the loaded body appears in context. --- ## Phase 1 — Bundled resources & scripts (level-3 disclosure) ',
  'metadata': {'source': '14-skills.md',
   'path': 'docs/14-skills.md',
   'ext': '.md',
   'doc_id': '14-skills.md',
   'chunk_index': 6}},
 {'id': '02-agent-loop-and-runtime.md#3',
  'score': 0.42326862338640736,
  'text': 'class class Pause: reason: str; payload: dict # HITL / external wait # middleware over a single step @dataclass class StepContext: agent: Agent; state: RunState; svc: Services; window: list[Message] | None = None class StepInterceptor(Protocol): async def __call__(self, ctx: StepContext, nxt: Callable

The magic returns a list of dicts (`id`, `score`, `text`, `metadata`) and
prints a formatted view via `rag.tools.format_documents` so the cell output is
readable. You can also just do `answer(...)` from `rag.generate` for a real
LLM-synthesized response.

## 2. MCP — connect, list, call

MCP magics hold per-kernel state. Each `%mcp_add` spawns a stdio subprocess
and connects via `MCPClient`. The subprocess lives until the kernel shuts down
or you call `%unload_ext openmate.jupyter_magic`.

### 2a. Connect to a server

In [4]:
# Use the in-repo fake Gmail server for an offline demo:
%mcp_add fake_gmail python3 servers/gmail/server.py

# Or, if you've done the OAuth dance (see .env.example), the real one:
# %mcp_add gmail npx -y @gongrzhe/server-gmail-autoauth-mcp


connected to 'fake_gmail' (python3…); 5 tools: ['gmail_search', 'gmail_get_message', 'gmail_get_thread', 'gmail_list_labels', 'gmail_create_draft']


### 2b. List what tools it exposes

In [5]:
%mcp_list fake_gmail


- gmail_search: Search the mailbox. Returns {items, next_page_token}; pass page_token to continue. Supports Gmail operators (is:unread, label:NAME, from:addr, newer_than:7d). Read-only and safe to retry.
- gmail_get_message: Fetch one message by id: decoded headers + plaintext body. Read-only.
- gmail_get_thread: Fetch a whole thread by id: its messages in chronological order. Read-only.
- gmail_list_labels: List mailbox labels with message counts. Read-only.
- gmail_create_draft: Create a DRAFT reply or new email (never sends). Returns the draft id. Use this to propose a reply for the user to review and send themselves.


[{'name': 'gmail_search',
  'remote_name': 'gmail_search',
  'description': 'Search the mailbox. Returns {items, next_page_token}; pass page_token to continue. Supports Gmail operators (is:unread, label:NAME, from:addr, newer_than:7d). Read-only and safe to retry.',
  'schema': {'properties': {'q': {'default': '',
     'title': 'Q',
     'type': 'string'},
    'max_results': {'default': 10, 'title': 'Max Results', 'type': 'integer'},
    'page_token': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Page Token'}},
   'title': 'gmail_searchArguments',
   'type': 'object'}},
 {'name': 'gmail_get_message',
  'remote_name': 'gmail_get_message',
  'description': 'Fetch one message by id: decoded headers + plaintext body. Read-only.',
  'schema': {'properties': {'id': {'title': 'Id', 'type': 'string'},
    'format': {'default': 'full', 'title': 'Format', 'type': 'string'}},
   'required': ['id'],
   'title': 'gmail_get_messageArguments',
   'type': 'obje

### 2c. Call one

In [6]:
%mcp_call fake_gmail list_messages {"max_results": 5}


Unknown tool: list_messages


CallToolResult(meta=None, content=[TextContent(type='text', text='Unknown tool: list_messages', annotations=None, meta=None)], structuredContent=None, isError=True)

## 3. Going further

- For a real LLM-synthesized answer, import `rag.generate.answer` directly:
  ```python
  from rag import build_embedder, build_retriever, answer
  from openmate.ports.model import default_model
  emb, store = build_embedder(), build_store("memory")
  # ... ingest ...
  retriever = build_retriever(emb, store)
  print(await answer("what is the agent loop?", retriever, default_model()))
  ```
- The magics are pure wrappers — anything in `rag/` or `openmate.adapters.tools.mcp_client`
  is reachable from a normal cell. Use the magics for the 80% case, drop to
  Python for the rest.
- The MCP server we just connected to lives as a subprocess under this kernel.
  Restart the kernel when you're done to clean it up.
